# 深度学习计算

## 层和块

In [2]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

X = torch.rand(2, 20)

net(X)

tensor([[-0.0044,  0.1737,  0.0678,  0.1166,  0.0816, -0.1436, -0.1122, -0.1220,
         -0.2291, -0.3018],
        [ 0.0696, -0.1923,  0.2183,  0.1431,  0.0307, -0.1054, -0.1099, -0.0840,
         -0.1437, -0.1695]], grad_fn=<AddmmBackward0>)

### 自定义块

In [3]:
class MLP(nn.Module):
    # 用模型参数声明层。这里，我们声明两个全连接的层
    def __init__(self):
        # 调用MLP的父类Module的构造函数来执行必要的初始化
        # 这样，在类实例化时也可以指定其他函数参数，例如模型参数params
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)

    # 定义模型的前向传播，即如何根据输入X返回所需的模型输出
    def forward(self, X):
        # 注意，这里我们使用ReLU的函数版本，其在nn.functional模块中定义
        return self.out(F.relu(self.hidden(X)))

In [4]:
net = MLP()
net(X)

tensor([[-0.0147,  0.1324, -0.0705, -0.1574, -0.0102,  0.0153, -0.2262, -0.0990,
         -0.0254, -0.0725],
        [ 0.0672, -0.0150, -0.0998, -0.1236, -0.0742,  0.0051, -0.1149, -0.2015,
          0.0779,  0.0226]], grad_fn=<AddmmBackward0>)

### 顺序块

_modules的主要优点是：在模块的参数初始化过程中，系统知道在_modules字典中查找需要初始化参数的⼦块。

In [6]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            # 这里，module是Module子类的一个实例。我们把它把保存在'Module'类的成员
            # 变量_modules中。_module的类型是OrderedDict
            self._modules[str(idx)] = module
    
    def forward(self, X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X

In [7]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[ 0.0778,  0.1070,  0.0773, -0.1237, -0.0993, -0.0746,  0.0877,  0.0754,
          0.2499, -0.0997],
        [-0.2676,  0.1421,  0.0155, -0.1965, -0.0202, -0.0876, -0.0558,  0.0047,
          0.2800, -0.2742]], grad_fn=<AddmmBackward0>)

### 在前向传播函数中执行代码

In [9]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机权重参数。因此其在训练期间保持不变
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        # 使用创建的常量参数以及relu和mm函数
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        # 复用全连接层。这相当于两个全连接层共享参数
        X = self.linear(X)
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()